+WQ# Cuaderno de Clase — GloVe y FastText

## Objetivo

Conocer dos alternativas a Word2Vec —GloVe y FastText— y comprender en qué se diferencian, con foco especial en la ventaja de FastText para manejar palabras fuera del vocabulario (OOV).

## Resultados de aprendizaje

Al terminar este cuaderno vas a poder:

- Explicar cómo GloVe construye vectores a partir de co-ocurrencias globales.
- Describir cómo FastText representa palabras usando n-gramas de caracteres.
- Usar vectores FastText pre-entrenados para buscar similitudes y manejar palabras OOV.
- Comparar el comportamiento de Word2Vec y FastText ante palabras desconocidas.
- Reflexionar sobre cómo evaluar la calidad de embeddings y detectar sesgos.

## Relación con cuadernos anteriores

En el cuaderno anterior trabajamos con Word2Vec, que asigna un vector a cada palabra completa vista durante el entrenamiento. Acá exploramos qué pasa cuando el modelo se encuentra con una palabra que nunca vio antes.


In [1]:
# Instalamos gensim si no está disponible
!pip install gensim -q



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Instalaciones e importaciones

Usamos `gensim` para cargar modelos FastText pre-entrenados, igual que hicimos con Word2Vec.


In [2]:
# 'gensim' para trabajar con embeddings pre-entrenados
import gensim

# 'KeyedVectors' para Word2Vec; 'FastText' para el formato de FastText
from gensim.models import KeyedVectors, FastText

# 'numpy' para operaciones numéricas
import numpy as np

# Silenciamos advertencias menores
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


# 2. Repaso Rápido: Word2Vec y sus limitaciones (OOV)

Recordemos:
*   Word2Vec (CBOW/Skip-gram) aprende embeddings prediciendo palabras en contextos locales.
*   Genera vectores densos que capturan semántica (similitud, analogías).
*   Usamos modelos pre-entrenados porque entrenarlos es costoso.

**Una limitación importante:** Word2Vec asigna un vector a cada palabra *completa* que vio durante el entrenamiento. Si en tiempo de uso aparece una palabra que **NO estaba en el vocabulario original** (Out-Of-Vocabulary - OOV):
*   Un error tipográfico ("computadorra").
*   Una palabra nueva o muy rara ("covid", "guasap").
*   Una variante morfológica no vista ("cantábamos").

**¿Qué pasa con Word2Vec?** ¡Generalmente da un error (`KeyError`) o asigna un vector genérico de "desconocido" (UNK - Unknown), perdiendo toda información específica! Esto es un problema en aplicaciones reales.

# 3. GloVe: Vectores Globales desde Co-ocurrencias

GloVe (Global Vectors for Word Representation) de Stanford es otra técnica popular para obtener embeddings.

*   **Enfoque Diferente:** En lugar de predecir contextos locales (como Word2Vec), GloVe se enfoca en las **estadísticas globales de co-ocurrencia** de palabras en todo el corpus.
*   **Idea Central:** La *relación* entre las probabilidades de co-ocurrencia de dos palabras con una tercera palabra "sonda" puede revelar información sobre su significado. Por ejemplo, la probabilidad de que "hielo" aparezca cerca de "sólido" será alta, mientras que cerca de "gas" será baja. La probabilidad de que "vapor" aparezca cerca de "gas" será alta y cerca de "sólido" baja. GloVe modela estas *proporciones* de probabilidades.
*   **Entrenamiento:** Construye una matriz gigante de co-ocurrencia (cuántas veces la palabra X aparece cerca de la palabra Y) y luego usa factorización de matrices para encontrar los vectores de palabras que mejor explican esa matriz.
*   **Resultado:** Embeddings densos similares a Word2Vec, a menudo muy buenos para tareas de similitud y analogía.
*   **Limitación OOV:** Al igual que Word2Vec, GloVe tradicionalmente opera a nivel de palabra completa, por lo que también sufre del problema de OOV.

*(Nota: Encontrar modelos GloVe pre-entrenados y de buena calidad para español puede ser más difícil que para Word2Vec o FastText. A menudo se distribuyen en formato texto).*

# 4. FastText: El Poder de las Subpalabras (¡Adiós OOV!)

FastText, desarrollado por Facebook AI Research (FAIR), es una extensión inteligente de Word2Vec (Skip-gram) que aborda directamente el problema OOV.

*   **¡La Gran Idea!:** FastText no trata las palabras como unidades indivisibles. Representa cada palabra como una **bolsa de n-gramas de caracteres**, además de la palabra completa.
    *   Ejemplo: Para la palabra "amarillo" y usando n=3 (trigramas):
        *   Se consideran n-gramas como: `<am`, `ama`, `mar`, `ari`, `ril`, `ill`, `llo`, `lo>` (con `<` y `>` marcando inicio y fin).
        *   También se considera la palabra completa: `<amarillo>`
*   **Aprendizaje:** El modelo aprende vectores no solo para las palabras completas, sino **para todos los n-gramas de caracteres** vistos en el corpus.
*   **Vector Final:** El vector de una palabra se obtiene **sumando los vectores de todos sus n-gramas de caracteres** (y el vector de la palabra completa si existe).

**¡La Ventaja Clave - Manejo de OOV!**
*   Si aparece una palabra nueva que no estaba en el vocabulario (ej: "amarillento"), FastText **aún puede construir un vector para ella**. ¿Cómo? Sumando los vectores de los n-gramas que sí conoce y que forman parte de la palabra nueva (ej: `<am`, `ama`, `mar`, `ari`, `ril`, `ill`, `lle`, `len`, `ent`, `nto`, `to>`).
*   El vector resultante captura información de las "partes" de la palabra, lo que a menudo resulta en una representación razonable incluso para palabras desconocidas.

**Otros Beneficios:**
*   Funciona muy bien para **lenguajes morfológicamente ricos** (como el español) donde muchas palabras comparten raíces y afijos (ej: "nación", "nacional", "nacionalidad"). Comparten n-gramas, sus vectores estarán relacionados.
*   Es más robusto a **errores tipográficos** ("computadorra" compartirá muchos n-gramas con "computadora").

# 5. Cargando Vectores FastText Pre-entrenados

Al igual que con Word2Vec, lo usual es usar modelos FastText pre-entrenados.

**Tarea:** Descargar los vectores pre-entrenados para **español** desde el sitio oficial de FastText: [https://fasttext.cc/docs/en/pretrained-vectors.html](https://fasttext.cc/docs/en/pretrained-vectors.html)

*   **Importante:** Descargar el archivo `.bin`. Este formato contiene no solo los vectores de las palabras del vocabulario, sino también la información de los n-gramas necesaria para calcular vectores de palabras OOV. El formato `.vec` es más pequeño pero pierde esta capacidad OOV.
*   El archivo `.bin` para español también es grande (varios GB). Anoten la ruta donde lo guardan.

**¡Reemplacen `'RUTA_AL_ARCHIVO_FASTTEXT.bin'` con la ruta real!**

## 5. Carga del modelo FastText pre-entrenado

El archivo `wiki.es.bin` es el modelo FastText entrenado sobre la Wikipedia en español. Está disponible en la carpeta `vectors/` de este módulo.

> **Nota técnica:** a diferencia del formato Word2Vec (.bin.gz), el archivo `.bin` de FastText contiene también la información de los n-gramas necesaria para calcular vectores de palabras OOV. Si usáramos el formato `.vec`, perderíamos esa capacidad.


In [4]:
import os

# Ruta al archivo de vectores FastText pre-entrenados
ruta_fasttext = r"C:\dominguez-micaela-belen-pln-1c-2026\010\002-PRA\vectors\wiki.es.bin"

# ruta_fasttext = r"E:\IFTS24-Cuadernos 2026\ifts24-lab-pln-2026\010_representaciones_semanticas_parte2\vectors\wiki.es.bin"

print(f"Ruta FastText configurada: {ruta_fasttext}")
print(f"El archivo existe: {os.path.exists(ruta_fasttext)}")

# También configuramos la ruta al modelo Word2Vec para poder comparar después
ruta_word2vec = os.path.join(os.path.dirname(os.path.abspath("__file__")), "vectors", "SBW-vectors-300-min5.bin.gz")
print(f"Ruta Word2Vec configurada: {ruta_word2vec}")
print(f"El archivo existe: {os.path.exists(ruta_word2vec)}")


Ruta FastText configurada: C:\dominguez-micaela-belen-pln-1c-2026\010\002-PRA\vectors\wiki.es.bin
El archivo existe: True
Ruta Word2Vec configurada: c:\dominguez-micaela-belen-pln-1c-2026\010\002-PRA\vectors\SBW-vectors-300-min5.bin.gz
El archivo existe: True


### Cargando el modelo

Este proceso puede tardar varios minutos: el archivo `wiki.es.bin` tiene más de 4 GB.

> **Para mirar:** ¿cuántas palabras tiene el vocabulario de FastText? Comparalo con el SBWC (Word2Vec) que cargamos en el cuaderno anterior.


In [5]:
# Cargamos el modelo FastText
# 'FastText.load_fasttext_format' entiende el formato nativo .bin de FastText
# que incluye tanto los vectores de palabras como los n-gramas de caracteres
try:
    print("Cargando vectores FastText... (puede tardar varios minutos)")
    fasttext_vectors = FastText.load_fasttext_format(ruta_fasttext)
    total_palabras_ft = len(fasttext_vectors.wv.index_to_key)
    dimension_ft = fasttext_vectors.wv.vector_size
    print(f"Vectores FastText cargados correctamente.")
    print(f"Vocabulario: {total_palabras_ft} palabras.")
    print(f"Dimensión de cada vector: {dimension_ft}.")

except FileNotFoundError:
    print(f"No se encontró el archivo en: {ruta_fasttext}")
    print("Verificá que el archivo esté en la carpeta 'vectors/' del módulo.")
    fasttext_vectors = None

except Exception as e:
    print(f"Error inesperado al cargar FastText: {e}")
    fasttext_vectors = None


Cargando vectores FastText... (puede tardar varios minutos)
Vectores FastText cargados correctamente.
Vocabulario: 985667 palabras.
Dimensión de cada vector: 300.


Tiene menos cantidad de palabras. Eso esta asociado por haber sido entrenados con corpus distintos


### Recargando Word2Vec para la comparación

Para comparar ambos modelos, necesitamos tener los dos cargados al mismo tiempo.


In [6]:
# Cargamos también el modelo Word2Vec para poder comparar después
try:
    print("Cargando vectores Word2Vec...")
    word2vec_vectors = KeyedVectors.load_word2vec_format(ruta_word2vec, binary=True)
    total_palabras_w2v = len(word2vec_vectors.index_to_key)
    print(f"Vectores Word2Vec cargados. Vocabulario: {total_palabras_w2v} palabras.")

except FileNotFoundError:
    print(f"No se encontró el archivo Word2Vec en: {ruta_word2vec}")
    word2vec_vectors = None

except Exception as e:
    print(f"Error al cargar Word2Vec: {e}")
    word2vec_vectors = None


Cargando vectores Word2Vec...
Vectores Word2Vec cargados. Vocabulario: 1692025 palabras.


## 6. Explorando FastText — similitud, analogías y OOV

Vamos a explorar el modelo FastText de la misma manera que hicimos con Word2Vec: similitudes y analogías. Después hacemos la prueba clave: palabras OOV.

> **Para mirar:** ¿los resultados de similitud y analogía son parecidos a los de Word2Vec? ¿Hay diferencias notables?


In [7]:
if fasttext_vectors is not None:
    print("Explorando FastText...")
    print()

    # Similitud entre palabras
    try:
        similares_rey = fasttext_vectors.wv.most_similar('rey', topn=5)
        print("Palabras más similares a 'rey' (FastText):")
        for palabra, puntaje in similares_rey:
            print(f"  {palabra}: {puntaje:.4f}")
    except Exception as e:
        print(f"Error buscando similares: {e}")

    print()

    # Analogía vectorial
    try:
        resultado_analogia = fasttext_vectors.wv.most_similar(
            positive=['rey', 'mujer'], negative=['hombre'], topn=1
        )
        print(f"Analogía rey - hombre + mujer = {resultado_analogia[0][0]} (puntaje: {resultado_analogia[0][1]:.4f})")
    except Exception as e:
        print(f"Error en la analogía: {e}")
else:
    print("El modelo FastText no está cargado.")


Explorando FastText...

Palabras más similares a 'rey' (FastText):
  monarca: 0.7459
  rey―: 0.6321
  exmonarca: 0.6311
  rey,: 0.6303
  reina: 0.6260

Analogía rey - hombre + mujer = reina (puntaje: 0.6612)


Los resultados de similitud son parecidos — en ambos aparece reina y monarca. La analogía también funciona igual en los dos. La diferencia notable es que FastText trae "ruido" como rey-: y rey,

## 7. La prueba clave: palabras OOV

Acá está la diferencia fundamental entre Word2Vec y FastText. Vamos a intentar obtener vectores para palabras que probablemente **no estén en el vocabulario** de ninguno de los dos modelos:

- Errores ortográficos: `"computadorra"`, `"programazion"`
- Neologismos o palabras recientes: `"guasap"`, `"covid"`
- Lenguaje inclusivo: `"estudiantes"` con terminaciones alternativas

> **Para mirar:** ¿Word2Vec lanza un error? ¿FastText logra construir un vector? ¿El vector de FastText tiene sentido semántico?


In [9]:
if word2vec_vectors is not None and fasttext_vectors is not None:
    print("Comparación de comportamiento ante palabras OOV:")
    print()

    # Palabras que probablemente no están en el vocabulario
    palabras_oov = ['computadorra', 'programazion', 'guasap', 'covid']

    for palabra in palabras_oov:
        esta_en_w2v = palabra in word2vec_vectors
        esta_en_ft = palabra in fasttext_vectors.wv

        print(f"Palabra: '{palabra}'")
        print(f"  Word2Vec — en vocabulario: {esta_en_w2v}")

        # Word2Vec: si no está, da KeyError
        if esta_en_w2v:
            vector_w2v = word2vec_vectors[palabra]
            print(f"  Word2Vec — vector obtenido: sí ({len(vector_w2v)} dimensiones)")
        else:
            print(f"  Word2Vec — vector obtenido: NO (KeyError si lo intentamos)")

        # FastText: aunque no esté en el vocabulario, puede construir un vector
        try:
            vector_ft = fasttext_vectors.wv[palabra]
            print(f"  FastText — vector obtenido: sí ({len(vector_ft)} dimensiones)")
        except Exception as e:
            print(f"  FastText — error: {e}")

        print()
else:
    print("Alguno de los modelos no está cargado.")


Comparación de comportamiento ante palabras OOV:

Palabra: 'computadorra'
  Word2Vec — en vocabulario: False
  Word2Vec — vector obtenido: NO (KeyError si lo intentamos)
  FastText — vector obtenido: sí (300 dimensiones)

Palabra: 'programazion'
  Word2Vec — en vocabulario: False
  Word2Vec — vector obtenido: NO (KeyError si lo intentamos)
  FastText — vector obtenido: sí (300 dimensiones)

Palabra: 'guasap'
  Word2Vec — en vocabulario: False
  Word2Vec — vector obtenido: NO (KeyError si lo intentamos)
  FastText — vector obtenido: sí (300 dimensiones)

Palabra: 'covid'
  Word2Vec — en vocabulario: False
  Word2Vec — vector obtenido: NO (KeyError si lo intentamos)
  FastText — vector obtenido: sí (300 dimensiones)



Si word2vec lanza error porque las palabras no estan en su vocabulario. Fast text construye un vector para todas ellas usando n gramas que conoce. Si tiene sentido semantico porque al compartir muchos n gramas el vector resultante deberia ser semanticamente parecido al de la palabra correcta por ejemplo computadorra muy cerca de computadora.

**Conclusión de la Comparativa:**

Deberían observar que para la mayoría (o todas) las palabras OOV, Word2Vec lanza un `KeyError`, mientras que FastText logra generar un vector y encontrar palabras similares (que pueden o no ser muy coherentes, dependiendo de qué tan "rara" sea la palabra OOV, pero *intenta* algo basado en sus partes).

**¿Cuándo usar FastText?**
*   Cuando trabajas con texto ruidoso (redes sociales, OCR) con typos.
*   Cuando tratas con lenguajes morfológicamente ricos (español, alemán, etc.).
*   Cuando esperas encontrar palabras nuevas o raras en tus datos de aplicación.
*   **En general, para español, FastText suele ser una opción más robusta y recomendable que Word2Vec o GloVe.**

**Desventaja:** Los modelos `.bin` de FastText son más grandes en disco y consumen más RAM porque almacenan información de los n-gramas.

# 8. Micro-Laboratorio (Ejercicio Práctico)

**Consigna:** (Asumiendo que `fasttext_vectors` y `word2vec_vectors` están cargados)

1.  **Comparación de Resultados:**
    *   Elegí 3 palabras que **sí** estén en ambos vocabularios (ej: 'gato', 'correr', 'inteligencia').
    *   Para cada palabra, obtener las 5 más similares usando `word2vec_vectors.most_similar()` y `fasttext_vectors.most_similar()`.
    *   Comparar las listas de similares. ¿Son idénticas? ¿Muy parecidas? ¿Diferentes? ¿Cuál les parece "mejor" o más coherente? Anotá observaciones.

2.  **Test OOV Exhaustivo:**
    *   Crear una lista propia de 10 palabras OOV. Incluyan:
        *   Errores tipográficos comunes (ej: "hobmre", "qeu", "dicimbre").
        *   Diminutivos/Aumentativos (ej: "perrito", "casita", "libraco").
        *   Formas verbales conjugadas (ej: "habíamos comido", "cantasteis").
        *   Palabras inventadas pero plausibles (ej: "tecnoestrés", "computofilia").
    *   Para **cada** palabra OOV de su lista:
        *   Verificar si da `KeyError` en `word2vec_vectors`.
        *   Obtener las 3 palabras más similares usando `fasttext_vectors`. Anotá los resultados. ¿Los similares que da FastText tienen algún sentido basado en las partes de la palabra OOV?

3.  **Discusión:**
    *   ¿En qué tipo de aplicación real (ej: un chatbot de atención al cliente, un sistema de recomendación de noticias, un corrector ortográfico) creen que la capacidad OOV de FastText marcaría una diferencia significativa respecto a usar Word2Vec? ¿Por qué?

In [11]:
palabras = ['gato', 'programador', 'Argentina']

for palabra in palabras:
    print(f"\n '{palabra}'")
    
    if palabra in word2vec_vectors:
        similares_w2v = word2vec_vectors.most_similar(palabra, topn=5)
        print(f"Word2Vec:")
        for p, puntaje in similares_w2v:
            print(f"  {p}: {puntaje:.4f}")
    
    print(f"FastText:")
    similares_ft = fasttext_vectors.wv.most_similar(palabra, topn=5)
    for p, puntaje in similares_ft:
        print(f"  {p}: {puntaje:.4f}")


 'gato'
Word2Vec:
  perro: 0.8182
  zorro: 0.7960
  oso: 0.7677
  conejo: 0.7547
  mono: 0.7344
FastText:
  gatos: 0.6868
  conejo: 0.6618
  perro: 0.6526
  gatito: 0.6358
  zorro: 0.6104

 'programador'
Word2Vec:
  desarrollador: 0.6927
  analista-programador: 0.6496
  interpretador: 0.6325
  Chris_Sawyer: 0.6270
  analista: 0.6205
FastText:
  desprogramador: 0.7881
  programadores: 0.7775
  programadorcccp: 0.7713
  diseñador/programador: 0.7614
  productor/programador: 0.7523

 'Argentina'
Word2Vec:
  Uruguay: 0.7776
  Chile: 0.7458
  Paraguay: 0.7389
  Buenos_Aires: 0.7258
  Bolivia: 0.7151
FastText:
  rgentina: 0.9560
  —argentina: 0.8798
  #argentina: 0.8349
  feargentina: 0.8252
  ,argentina: 0.8045


1.word3vec da resultados mas limpios y semanticamente .coherentes para palabras que ya estan en su vocabulario diferente a fasttext que lo que hace es traer variante con mucho ruido  como simbiolos hashtag lo que seria una desventaja.

In [12]:
palabras_oov = [
    'hobmre', 'qeu', 'dicimbre',      # errores tipográficos
    'perrito', 'casita', 'libraco',    # diminutivos/aumentativos
    'cantasteis', 'habíamos',          # formas verbales
    'tecnoestrés', 'clauding'          # palabras inventadas
]

for palabra in palabras_oov:
    esta_en_w2v = palabra in word2vec_vectors
    print(f"\nPalabra: '{palabra}'")
    print(f"  Word2Vec — en vocabulario: {esta_en_w2v}")
    
    try:
        similares_ft = fasttext_vectors.wv.most_similar(palabra, topn=3)
        print(f"  FastText — similares:")
        for p, puntaje in similares_ft:
            print(f"    {p}: {puntaje:.4f}")
    except Exception as e:
        print(f"  FastText — error: {e}")


Palabra: 'hobmre'
  Word2Vec — en vocabulario: False
  FastText — similares:
    nobmre: 0.6313
    ademre: 0.5202
    demre: 0.5149

Palabra: 'qeu'
  Word2Vec — en vocabulario: True
  FastText — similares:
    qeuab: 0.6885
    qeuad: 0.6671
    wikisilki/pi: 0.6111

Palabra: 'dicimbre'
  Word2Vec — en vocabulario: True
  FastText — similares:
    cimbres: 0.6493
    cimbro: 0.6210
    cimbros: 0.5852

Palabra: 'perrito'
  Word2Vec — en vocabulario: True
  FastText — similares:
    perritos: 0.8241
    perrita: 0.7756
    perritas: 0.7067

Palabra: 'casita'
  Word2Vec — en vocabulario: True
  FastText — similares:
    casitas: 0.7595
    casitagua: 0.6553
    asita: 0.6545

Palabra: 'libraco'
  Word2Vec — en vocabulario: True
  FastText — similares:
    libracos: 0.8515
    libración: 0.6721
    libraciones: 0.6215

Palabra: 'cantasteis'
  Word2Vec — en vocabulario: False
  FastText — similares:
    cantaste: 0.8619
    fuisteis: 0.7057
    gusteis: 0.7031

Palabra: 'habíamos'
  Word

2. Fasttext construye un vector donde word2vec no puede. Los mas coherentes son los de canrasteus habiamso y clauding. Los menos coherentes son hobmre y qeu donde los n gramas no fueron suficientes para inferir el significado reafl.

3. En un chatbot donde un usuario suele cometer errores de tipeo, abreviaciones. fast text trataria de entender mejor el mensaje


# 9. Brainstorming: Evaluación y Detección de Sesgos

Hemos visto diferentes formas de crear embeddings (Word2Vec, GloVe, FastText) y cómo usarlos. Pero, ¿cómo sabemos si un conjunto de embeddings es "bueno"? ¿Y cómo abordamos el problema de los sesgos que mencionamos en clases anteriores?

**Pregunta:** **¿Cómo podemos evaluar la calidad de los word embeddings y detectar posibles sesgos?**

*   **Evaluación Intrínseca:**
    *   Medir qué tan bien funcionan los embeddings en tareas específicas relacionadas con las palabras mismas, **sin** una aplicación final.
    *   Ejemplos:
        *   **Similitud de Palabras:** Comparar la similitud coseno entre pares de palabras según los embeddings, con juicios de similitud dados por humanos (datasets estándar como WordSim-353, SimLex-999, adaptados o creados para español).
        *   **Tareas de Analogía:** Ver qué tan bien resuelven analogías (`rey - hombre + mujer = ?`). Hay datasets estándar de analogías (Google Analogies, BATS).
*   **Evaluación Extrínseca:**
    *   Usar los embeddings como **características de entrada (features)** para un modelo de PLN más complejo que resuelve una tarea final (downstream task).
    *   Ejemplos: Clasificación de sentimientos, reconocimiento de entidades nombradas, clasificación de temas.
    *   Medir si usar estos embeddings mejora el rendimiento (precisión, F1-score, etc.) del sistema final comparado con no usarlos o usar otros embeddings. **Esta suele ser la evaluación más relevante en la práctica.**
*   **Detección de Sesgos:**
    *   **Analogías Específicas:** Crear analogías diseñadas para revelar sesgos sociales (género-profesión, raza-sentimiento, etc.). `programador - hombre + mujer = ?`.
    *   **Pruebas de Asociación Implícita (como WEAT - Word Embedding Association Test):** Miden matemáticamente qué tan asociados están ciertos grupos de palabras (ej: nombres masculinos vs femeninos) con ciertos atributos (ej: carreras científicas vs artísticas, adjetivos positivos vs negativos) dentro del espacio de embeddings.
    *   **Visualización:** A veces, proyectar grupos específicos (hombres/mujeres, profesiones) a 2D puede revelar agrupaciones o direcciones sesgadas.
    *   **Auditoría de Vecinos Cercanos:** Ver los `most_similar` para palabras sensibles.

**Una vez detectado el sesgo, ¿qué hacemos?** ¿Existen técnicas para mitigarlo ("debiasing")? ¿Es mejor buscar/crear corpus menos sesgados? ¿O simplemente ser transparentes sobre los sesgos del modelo?

**(Discusión en grupo)**

Una vez detectado el sesgo hay tres opciones:Una vez detectado el sesgo hay tres opciones:

Debiasing: técnicas matemáticas que modifican los vectores para reducir el sesgo, por ejemplo neutralizando la dirección género en el espacio vectorial.
Mejorar el corpus: curar mejor los datos de entrenamiento para que sean más diversos y representativos.
Transparencia: si no se puede eliminar, al menos documentar y advertir sobre los sesgos del modelo para que quien lo use sea consciente.

# GUÍA DE ESTUDIO - FASTTEXT Y GLOVE

## Preguntas y Respuestas Clave

### **Innovaciones Arquitectónicas**

**P: ¿Cuál es la innovación principal de FastText sobre Word2Vec?**  
R: FastText representa palabras como suma de n-gramas de caracteres, no como unidades indivisibles. Esto permite generar vectores para palabras nunca vistas (OOV).

**P: ¿Cómo funciona FastText con palabras desconocidas?**  
R: Descompone la palabra en n-gramas ("amarillo" → "<am", "ama", "mar", etc.) y suma los vectores de los n-gramas conocidos para generar un vector para la palabra completa.

**P: ¿En qué se diferencia GloVe de Word2Vec conceptualmente?**  
R: Word2Vec predice contextos locales. GloVe usa estadísticas globales de co-ocurrencia de todo el corpus para factorizar una matriz de co-ocurrencias.

### **Ventajas Específicas**

**P: ¿Por qué FastText es especialmente bueno para español?**  
R: El español es morfológicamente rico (muchas conjugaciones, derivaciones). FastText captura estas relaciones morfológicas a través de n-gramas compartidos.

**P: ¿Qué ventajas tiene FastText para texto ruidoso?**  
R: Es robusto a errores tipográficos porque "computadorra" comparte muchos n-gramas con "computadora", generando un vector similar.

**P: ¿Cuándo elegir GloVe sobre Word2Vec?**  
R: Cuando necesitas mejor aprovechamiento de estadísticas corpus-wide o tienes computational budget limitado para múltiples pasadas sobre el corpus.

### **Implementación Práctica**

**P: ¿Por qué es importante descargar archivos .bin de FastText?**  
R: El formato .bin incluye información de n-gramas necesaria para manejar OOV. El formato .vec solo tiene vectores de palabras conocidas.

**P: ¿Qué indica que una palabra no está en vocabulario explícito pero FastText puede procesarla?**  
R: `palabra in fasttext_vectors` devuelve False, pero `fasttext_vectors[palabra]` aún funciona generando un vector dinámicamente.

### **Evaluación y Comparación**

**P: ¿Cómo evaluar si FastText produce mejores embeddings que Word2Vec?**  
R: Evaluación intrínseca (tareas de similitud, analogías) y extrínseca (performance en tareas downstream como clasificación de texto).

**P: ¿Qué trade-offs tiene FastText?**  
R: Pro: maneja OOV, mejor morfología. Contra: mayor tamaño de modelo, mayor uso de RAM, menor interpretabilidad de n-gramas.

### **Casos de Uso**

**P: ¿En qué aplicaciones sería crítica la capacidad OOV de FastText?**  
R: Redes sociales (jerga, typos), análisis de texto histórico (ortografía variable), corpus especializados (terminología técnica), lenguas con poca documentación.

**P: ¿Cómo manejaría lenguaje inclusivo con FastText?**  
R: "Estudiantx" se procesaría usando n-gramas de "estudiante" + sufijo, capturando la relación semántica automáticamente.

## Puntos Clave para Recordar

1. **FastText = Word2Vec + n-gramas de caracteres**
2. **OOV no es problema** con FastText (vs KeyError en Word2Vec)
3. **GloVe = estadísticas globales** vs predicción local
4. **Para español: FastText > Word2Vec** por morfología rica
5. **Trade-off: capacidad vs complejidad computacional**
6. **Archivos .bin cruciales** para funcionalidad OOV

## Consideraciones Importantes

- FastText puede generar vectores para cualquier string, incluso nonsense
- La calidad del vector OOV depende de qué tan informativos sean los n-gramas
- GloVe puede ser difícil de encontrar pre-entrenado para español
- Modelos más grandes requieren más RAM y tiempo de carga
- No todos los n-gramas son igualmente informativos

## Conexión con Próxima Clase

Ahora conocen múltiples arquitecturas de embeddings. La próxima clase integrará todos estos métodos en **pipelines prácticos**, comparando su rendimiento en tareas reales y decidiendo cuándo usar cada uno.

---
*Consejo: Testa FastText con argentinismos o errores de tipeo comunes en WhatsApp. ¿Genera vectores sensatos para "kjjjj" o "messirve"?*

## Próximos pasos

Ya conocemos los embeddings estáticos más importantes: Word2Vec, GloVe y FastText. En el siguiente cuaderno vamos a dar un salto importante hacia los **embeddings de oraciones con modelos de lenguaje modernos** (sentence transformers), que resuelven otra limitación: la falta de contexto. Una misma palabra como "banco" tiene el mismo vector en todos estos modelos, sin importar si habla de un banco de plaza o de un banco financiero.
